Exploratory Data Analysis

In [3]:
# Libraries Needed for EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing, model_selection
from scipy import stats

In [ ]:
import os


RUTA = os.path.join('..', 'Raw Data')

df_sector = pd.read_excel(
    os.path.join(RUTA, 'inversion_sector.xlsx'),
    sheet_name='IED por actividad trim'
)
df_pais = pd.read_excel(
    os.path.join(RUTA, 'inversion_pais.xlsx'),
    sheet_name='IED por país trim'
)

# 1. Carga de bases de datos de IED (Y y Z)

# Limpieza y estructuración de la matriz sectorial (Y)
# La fila 0 contiene los identificadores de trimestres (2022-1 a 2025-4)
trimestres = df_sector.iloc[0, 1:].values
df_sec_data = df_sector.iloc[1:].copy()
df_sec_data.columns = ['Actividad_Economica'] + list(trimestres)

print('--- MATRIZ SECTORIAL DE IED (Y) CARGADA EXITOSAMENTE ---')
print(f'Total de sectores analizados: {len(df_sec_data)}')
print(f'Horizonte temporal: {trimestres[0]} al {trimestres[-1]} (N = 16)')

# 2. Cálculo de Estadísticos Descriptivos por Sector (Objetivo 1)
desc_results = []
for idx, row in df_sec_data.iterrows():
  sector = row['Actividad_Economica']
  valores = pd.to_numeric(row.iloc[1:], errors='coerce')

  # Pruebas de normalidad Shapiro-Wilk (Requisito metodológico Sección 4.4.2)
  shapiro_stat, shapiro_p = stats.shapiro(valores.dropna())

  desc_results.append({
      'Sector': sector,
      'Media': round(valores.mean(), 2),
      'Mediana': round(valores.median(), 2),
      'Mínimo': round(valores.min(), 2),
      'Máximo': round(valores.max(), 2),
      'Desv. Estándar': round(valores.std(), 2),
      'Shapiro_p_value': round(shapiro_p, 4),
  })

df_desc = pd.DataFrame(desc_results)
print('\n--- ESTADÍSTICOS DESCRIPTIVOS Y PRUEBA DE NORMALIDAD (SHAPIRO-WILK) ---')
print(df_desc.to_string(index=False))

# 3. Cálculo de la Variable de Control Z: Concentración Geográfica (EE. UU.)
df_pais_data = df_pais.iloc[1:].copy()
df_pais_data.columns = ['País'] + list(trimestres)

# Extraer fila de Estados Unidos y fila de Total (si aplica, o calcular suma)
eeuu_row = df_pais_data[df_pais_data['País'].str.contains('Estados Unidos', case=False, na=False)]

if not eeuu_row.empty:
  valores_eeuu = pd.to_numeric(eeuu_row.iloc[0, 1:], errors='coerce')
  print('\n--- VARIABLE DE CONTROL Z (ESTADOS UNIDOS) ---')
  print(f'Flujo trimestral promedio desde EE. UU.: {round(valores_eeuu.mean(), 2)} M USD')

--- MATRIZ SECTORIAL DE IED (Y) CARGADA EXITOSAMENTE ---
Total de sectores analizados: 10
Horizonte temporal: 2022-1 al 2025-4 (N = 16)

--- ESTADÍSTICOS DESCRIPTIVOS Y PRUEBA DE NORMALIDAD (SHAPIRO-WILK) ---
              Sector  Media  Mediana  Mínimo  Máximo  Desv. Estándar  Shapiro_p_value
             Turismo 300.32   313.00   229.3   342.8           32.86           0.0279
Comercio / Industria 145.62   158.05   -50.7   257.7           75.50           0.1185
  Telecomunicaciones  10.76     6.70   -63.6    78.2           43.17           0.4420
             Energía 263.99   282.00    70.8   412.6          101.99           0.6470
          Financiero  41.72    40.80    22.4    65.2           10.23           0.8017
       Zonas Francas  97.99    97.25    77.5   129.9           13.62           0.6221
              Minero  59.40    82.65  -301.1   252.2          125.55           0.0105
        Inmobiliario 178.59   191.00   113.9   249.3           41.62           0.2932
          Transpo

C:\Users\Sam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_morestats.py:2199: RuntimeWarning: invalid value encountered in scalar divide
  return num / den


In [ ]:

# Preparar datos numéricos (sectores en filas, trimestres en columnas)
df_num = df_sec_data.set_index('Actividad_Economica').apply(pd.to_numeric, errors='coerce')
df_num.columns = df_num.columns.astype(str)

# Si hay una fila de total, exclúyela para no distorsionar la escala
df_num = df_num[~df_num.index.str.contains('total', case=False, na=False)]

os.makedirs(os.path.join('..', 'Plots'), exist_ok=True)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for sector, fila in df_num.iterrows():
    ax.plot(fila.index, fila.values, marker='o', markersize=3, label=sector)

ax.set_title('IED por sector, 2022-1 a 2025-4')
ax.set_ylabel('Millones de USD')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'Plots', 'ied_sector_serie.png'), dpi=300)
plt.show()